# Melanin binding model

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, RDLogger
from rdkit.Chem import MACCSkeys, AllChem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.*')

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score,
)
from sklearn.metrics import (
    classification_report, f1_score, roc_curve, auc, roc_auc_score,
    accuracy_score,
)

import joblib

# Single source of truth for every split, model fit and figure below.
RANDOM_STATE = 42
N_SEEDS = 5


## Load dataset

The raw source is Jakubiak et al. (melanin binding, in vitro fraction-unbound
assay). We reconstruct the same `Class` definition used originally:
**Class 1 = fraction unbound &ge; 1%** (weak or no significant melanin
binding, majority class), **Class 0 = fraction unbound &lt; 1%** (strong
melanin binder, minority class).

Note the polarity: strong binding is the pharmacologically desirable property
here, and it is the class labelled **0**. Consumers of the saved model must
therefore read `predict_proba(X)[:, 0]` to obtain the probability of strong
binding used in the composite ocular reward.


In [ ]:
raw = pd.read_csv('../../data/ocular/melanin_raw.csv')
col = 'Melanin binding Fraction unbound (%)'

raw = raw.dropna(subset=['SMILES']).reset_index(drop=True)
raw['Class'] = (raw[col] >= 1).astype(int)
df = raw[['SMILES', 'Class']].copy()

print('Total compounds:', len(df))
print('Duplicated SMILES:', df['SMILES'].duplicated().sum())
print()
print('Class 1 (>=1% unbound, weak/no binding):', int((df['Class'] == 1).sum()))
print('Class 0 (<1% unbound, strong binding):  ', int((df['Class'] == 0).sum()))


### Dataset balancing

The majority class is downsampled to 200 compounds with a fixed seed, so the
balanced set of 373 compounds reported in the manuscript can be reproduced
exactly.


In [ ]:
df_1 = df.loc[df['Class'] == 1]
df_0 = df.loc[df['Class'] == 0]

df_balanced = pd.concat([
    df_1.sample(n=200, random_state=RANDOM_STATE),
    df_0,
]).reset_index(drop=True)

print('Balanced set:', len(df_balanced))
print(df_balanced['Class'].value_counts().to_dict())


### Convert SMILES to MACCS fingerprints

In [ ]:
header = ['bit' + str(i) for i in range(167)]


def smiles_to_maccs(smiles_list):
    rows, keep = [], []
    for i, s in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            continue
        rows.append(list(MACCSkeys.GenMACCSKeys(mol).ToBitString()))
        keep.append(i)
    out = pd.DataFrame(rows, columns=header).astype(int)
    out.insert(0, 'SMILES', [smiles_list[i] for i in keep])
    return out


df_fp = smiles_to_maccs(df_balanced['SMILES'].tolist())
df_fp = df_fp.merge(df_balanced, on='SMILES', how='left')

X = df_fp[header]
y = df_fp['Class'].values
print('fingerprint matrix:', X.shape)


## Train/test split

One stratified split at `RANDOM_STATE` is used for the grid search, the ROC
curve, the feature-importance plot and the saved model.


In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)
print(f"n_train = {len(ytrain)}   n_test = {len(ytest)}")
print(f"train_pos = {ytrain.mean():.2f}   test_pos = {ytest.mean():.2f}")


## Hyperparameter search

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

param_grid = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
}

base_estimator = ExtraTreesClassifier(random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=base_estimator,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(Xtrain, ytrain)

print("best params:", grid_search.best_params_)
print(f"best CV F1 = {grid_search.best_score_:.3f}")


### Does tuning beat the defaults?

Compared on cross-validated F1 within the training partition, never on the
test set.


In [ ]:
default_cv_f1 = cross_val_score(
    clone(base_estimator), Xtrain, ytrain,
    scoring='f1', cv=cv, n_jobs=-1,
).mean()

print(f"default CV F1 = {default_cv_f1:.3f}   "
      f"tuned CV F1 = {grid_search.best_score_:.3f}")

if grid_search.best_score_ >= default_cv_f1:
    final_model = clone(grid_search.best_estimator_)
    final_params = grid_search.best_params_
    print("-> using tuned hyperparameters")
else:
    final_model = clone(base_estimator)
    final_params = "ExtraTrees defaults"
    print("-> tuning did not improve on the defaults; using defaults")

final_model.fit(Xtrain, ytrain)
print()
print(classification_report(ytest, final_model.predict(Xtest)))
print("hyperparameters for Table 7:", final_params)


## Test-set performance

In [ ]:
proba_test = final_model.predict_proba(Xtest)[:, 1]
fpr, tpr, _ = roc_curve(ytest, proba_test)
roc_auc = auc(fpr, tpr)
print(f"test ROC-AUC = {roc_auc:.3f}")

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#A2C0D9', linewidth=3,
         label='ROC curve (area = {:.2f})'.format(roc_auc))
plt.plot([0, 1], [0, 1], color='#A2C0D9', linestyle='--', linewidth=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=20)
plt.ylabel('True Positive Rate', fontsize=20)
plt.legend(loc='lower right', fontsize=20)
plt.tick_params(axis='both', labelsize=16)
plt.grid()
plt.savefig('mel_ML.svg', dpi=700, bbox_inches='tight')
plt.show()


## Feature importance

In [ ]:
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': final_model.feature_importances_,
}).sort_values(by='Importance', ascending=False)

top5_df = feature_importance_df.head(5)
print(top5_df.to_string(index=False))

sns.set_style("whitegrid")
plt.figure(figsize=(8, 6))
sns.barplot(x='Importance', y='Feature', data=top5_df,
            color='#A2C0D9', alpha=0.9)
plt.xlabel('Importance Score', fontsize=22)
plt.ylabel('')
plt.xlim(0, 0.12)
plt.tick_params(axis='x', labelsize=20)
plt.tick_params(axis='y', labelsize=20)
plt.savefig('mel_FI.svg', dpi=700, bbox_inches='tight')
plt.show()


## Save model

In [ ]:
joblib.dump(final_model, '../../models/melanin.pkl')
print('saved:', type(final_model).__name__, final_params)


## Robustness under scaffold splitting

Evaluated on the balanced set, with the same hyperparameters as the saved
model, so that Table 6 and Figure S1 describe the same classifier.


In [ ]:
def morgan_fp(smi, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)


def scaffold_of(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
    except Exception:
        return None


def scaffold_split(smiles_list, test_size=0.2, seed=RANDOM_STATE):
    """Group compounds by Murcko scaffold and assign whole groups to one
    partition. Groups are filled into the test set largest-first, so the test
    set holds the most populated scaffold families and the training set keeps
    the long tail of singletons."""
    scaffolds = {}
    for idx, smi in enumerate(smiles_list):
        scaffolds.setdefault(scaffold_of(smi), []).append(idx)

    groups = list(scaffolds.values())
    rng = np.random.RandomState(seed)
    groups = [groups[i] for i in rng.permutation(len(groups))]
    groups.sort(key=len, reverse=True)

    n_test_target = int(round(len(smiles_list) * test_size))
    test_idx, train_idx = [], []
    for g in groups:
        if len(test_idx) < n_test_target:
            test_idx.extend(g)
        else:
            train_idx.extend(g)
    return np.array(train_idx), np.array(test_idx)


def nn_tanimoto_stats(train_smiles, test_smiles):
    train_fps = [f for f in (morgan_fp(s) for s in train_smiles) if f is not None]
    sims = []
    for s in test_smiles:
        fp = morgan_fp(s)
        if fp is None or not train_fps:
            continue
        sims.append(max(DataStructs.BulkTanimotoSimilarity(fp, train_fps)))
    sims = np.array(sims)
    return {
        'mean_NN_Tanimoto': float(np.mean(sims)) if len(sims) else np.nan,
        'median_NN_Tanimoto': float(np.median(sims)) if len(sims) else np.nan,
        'frac_test_with_NN>0.85': float(np.mean(sims > 0.85)) if len(sims) else np.nan,
        'frac_test_with_NN>0.4': float(np.mean(sims > 0.4)) if len(sims) else np.nan,
    }


In [ ]:
smiles_all = df_fp['SMILES'].values
X_all = X.values
y_all = y
idx_all = np.arange(len(y_all))

tr_idx, te_idx = train_test_split(
    idx_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all,
)


In [ ]:
def eval_split(train_idx, test_idx, label):
    """Evaluate the final model's configuration on a given partition."""
    m = clone(final_model)
    m.fit(X_all[train_idx], y_all[train_idx])
    proba = m.predict_proba(X_all[test_idx])[:, 1]
    pred = m.predict(X_all[test_idx])
    sims = nn_tanimoto_stats(smiles_all[train_idx], smiles_all[test_idx])

    row = dict(
        label=label,
        n_train=len(train_idx), n_test=len(test_idx),
        train_pos=float(y_all[train_idx].mean()),
        test_pos=float(y_all[test_idx].mean()),
        auc=roc_auc_score(y_all[test_idx], proba),
        f1=f1_score(y_all[test_idx], pred),
        acc=accuracy_score(y_all[test_idx], pred),
        **sims,
    )
    print(f"--- {label} ---")
    print(f"  n_train={row['n_train']} n_test={row['n_test']}  "
          f"train_pos={row['train_pos']:.2f} test_pos={row['test_pos']:.2f}")
    print(f"  ROC-AUC={row['auc']:.3f}  F1={row['f1']:.3f}  Acc={row['acc']:.3f}")
    print(f"  NN-Tanimoto (test->train): mean={row['mean_NN_Tanimoto']:.3f} "
          f"median={row['median_NN_Tanimoto']:.3f}  "
          f"frac>0.85={row['frac_test_with_NN>0.85']:.2f}  "
          f"frac>0.4={row['frac_test_with_NN>0.4']:.2f}")
    return row


results = [eval_split(tr_idx, te_idx, "Random")]

seed_aucs = []
for seed in range(N_SEEDS):
    tr_s, te_s = train_test_split(idx_all, test_size=0.2,
                                  random_state=seed, stratify=y_all)
    m = clone(final_model).fit(X_all[tr_s], y_all[tr_s])
    seed_aucs.append(
        roc_auc_score(y_all[te_s], m.predict_proba(X_all[te_s])[:, 1]))

print(f"\nRandom split ROC-AUC over {N_SEEDS} seeds: "
      f"{np.mean(seed_aucs):.3f} +/- {np.std(seed_aucs):.3f}  "
      f"{[round(a, 3) for a in seed_aucs]}\n")

tr_sc, te_sc = scaffold_split(smiles_all, test_size=0.2, seed=RANDOM_STATE)
results.append(eval_split(tr_sc, te_sc, "Bemis-Murcko scaffold"))

table6 = pd.DataFrame(results)[
    ['label', 'n_train', 'n_test', 'auc',
     'mean_NN_Tanimoto', 'frac_test_with_NN>0.85']
].round(3)

print("\n=== numbers for Table 6 (MB rows) ===")
print(table6.to_string(index=False))
print(f"\n=== hyperparameters for Table 7 ===\n{final_params}")
print(f"\n=== ROC-AUC quoted in the text and in Figure S1 ===\n{roc_auc:.3f}")


## Numbers used in the manuscript

Every value below comes from the same pipeline: one stratified split at
`RANDOM_STATE = 42` and the single `final_model`.

| Where | Value |
|---|---|
| Table 6, MB Random | `n_train`, `n_test`, `auc`, `mean_NN_Tanimoto` printed above |
| Table 6, MB scaffold | same, from the scaffold row |
| Table 7, MB row | `final_params` printed above |
| Figure S1A | `mel_ML.svg`, drawn from `final_model` |
| Figure S1B | `mel_FI.svg`, drawn from `final_model` |
| `models/melanin.pkl` | `final_model` |
